# Lab 2: LLM and Transformer Fine-Tuning

The notebook is organized as eight consecutive blocks (Stages 0–7) that carry conversational supervision from OASST1 through supervised fine-tuning, quantitative evaluation, decoding analysis, and a compact interactive client built around a small causal transformer.

Stages 0–3 establish reproducible inputs and shared training machinery. Stage 0 materializes cache and logging directories and resolves device placement before any download so artifact paths stay stable when cells are re-executed out of order; Colab optionally installs pinned dependencies from requirements.txt. Stage 1 ingests OASST1, emits prompt–response rows, and fixes a 70/15/15 train/validation/test partition so later metrics are not computed on text already seen during model selection. Stage 2 applies the formatting and tokenization contract used at training time, plots corpus diagnostics, and scores cleaning variants with frozen-base proxy perplexity to quantify preprocessing-induced covariate shift prior to adaptation. Stage 3 concentrates dataset materialization, configuration hashing to compressed pickles, MLflow logging without duplicating weight blobs, and the early-stopped fine-tuning routine reused by every experiment cell.

Stages 4–7 interpret behavior under controlled changes. Stage 4 factorially swaps optimizers (AdamW, SGD, RMSprop) and schedulers (none versus linear warmup with linear decay) while holding architecture, data budget, and stopping rules fixed, which isolates first-order optimization effects. Stage 5 ranks checkpoints by test perplexity, then layers BLEU and ROUGE on held-out generations because strong likelihood need not coincide with acceptable surface wording. Stage 6 varies tokenizer max_length, max_new_tokens, top_k, and top_p in logged sweeps, runs a small multi-topic prompt battery for qualitative comparison, and appends structured lines to generation_interaction_log.txt. Stage 7 emits Streamlit with matching decode controls and records every chat turn as JSON lines in chat_log.txt for the same style of offline review.


## Stage 0. Setup and Reproducibility

Result trees for pickles, MLflow runs, and model archives are created relative to ROOT before any dataset or checkpoint I/O, which avoids partial writes when the notebook is run non-linearly. Colab is detected via google.colab and may install requirements.txt once; otherwise execution only binds DEVICE, keeping tensor placement orthogonal to data-processing code.


In [1]:
import subprocess
import sys
from pathlib import Path


ROOT = Path.cwd()
RESULTS_CACHE_DIR = ROOT / "results_cache"
MLFLOW_TRACKING_DIR = RESULTS_CACHE_DIR / "mlruns"
ARTIFACTS_DIR = RESULTS_CACHE_DIR / "artifacts"
MODEL_CACHE_DIR = RESULTS_CACHE_DIR / "models"

for p in [RESULTS_CACHE_DIR, MLFLOW_TRACKING_DIR, ARTIFACTS_DIR, MODEL_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

IS_COLAB = "google.colab" in sys.modules
REQUIREMENTS_PATH = ROOT / "requirements.txt"

if IS_COLAB and REQUIREMENTS_PATH.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_PATH)])

In [2]:
import gzip
import hashlib
import json
import math
import pickle
import random
import re
import shutil
import tarfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.optim as optim
from datasets import Dataset, DatasetDict, load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

import evaluate

SEED = 42


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed(SEED)
DEVICE = get_device()

DEVICE

/Users/lopatenko/Desktop/itmo/itmo-ml-2025/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='mps')

In [3]:
MLFLOW_EXPERIMENT_NAME = "dl-lab-2-llm"

try:
    import mlflow
    MLFLOW_AVAILABLE = True
except ImportError:
    mlflow = None
    MLFLOW_AVAILABLE = False

USE_MLFLOW = MLFLOW_AVAILABLE

if USE_MLFLOW:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_DIR.resolve().as_uri())
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)


@dataclass(frozen=True)
class TrainingConfig:
    """Configuration for one fine-tuning experiment.

    Attributes:
        model_name: Hugging Face model identifier.
        dataset_name: Dataset id for logging and cache identity.
        max_length: Max token length for training chunks.
        train_examples: Max train rows used for one run.
        val_examples: Max validation rows used for one run.
        test_examples: Max test rows used for one run.
        num_epochs: Number of training epochs.
        batch_size: Mini-batch size.
        learning_rate: Initial learning rate.
        optimizer_name: Optimizer name: adamw, sgd, rmsprop.
        scheduler_name: Scheduler mode: none, linear_warmup.
        warmup_ratio: Warmup share of total update steps.
        weight_decay: Weight decay value.
        grad_accum_steps: Gradient accumulation factor.
        early_stopping_patience: Number of epochs without validation improvement before stop.
        generation_max_new_tokens: Max generated tokens for eval generation.
    """

    model_name: str = "distilgpt2"
    dataset_name: str = "OpenAssistant/oasst1"
    max_length: int = 128
    train_examples: int = 8000
    val_examples: int = 1200
    test_examples: int = 1200
    num_epochs: int = 8
    batch_size: int = 8
    learning_rate: float = 5e-5
    optimizer_name: str = "adamw"
    scheduler_name: str = "none"
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    grad_accum_steps: int = 1
    early_stopping_patience: int = 3
    generation_max_new_tokens: int = 64


def cfg_to_cache_key(cfg: TrainingConfig, suffix: str = "") -> str:
    payload = asdict(cfg)
    payload["suffix"] = suffix
    raw = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha1(raw).hexdigest()


def cache_path_for_cfg(cfg: TrainingConfig, suffix: str = "") -> Path:
    """Return path to gzip-compressed pickle cache for this config."""
    return RESULTS_CACHE_DIR / f"{cfg_to_cache_key(cfg, suffix)}.pkl.gz"


def save_result(result: Any, cfg: TrainingConfig, suffix: str = "") -> Any:
    path = cache_path_for_cfg(cfg, suffix)
    with gzip.open(path, "wb", compresslevel=6) as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)

    if USE_MLFLOW:
        # Log params + scalar metrics only. Do not log the pickle as an MLflow artifact:
        # it duplicates results_cache on disk. Fast reruns still use the gzip cache file above.
        with mlflow.start_run(tags={"cache_key": cfg_to_cache_key(cfg, suffix), "suffix": suffix or "default"}):
            mlflow.log_params(asdict(cfg))
            mlflow.log_param("cache_path_gz", str(path.resolve()))
            if isinstance(result, dict):
                for k, v in result.items():
                    if isinstance(v, (int, float)) and math.isfinite(v):
                        mlflow.log_metric(k, float(v))

    return result


def load_result(cfg: TrainingConfig, suffix: str = "") -> Optional[Any]:
    path_gz = cache_path_for_cfg(cfg, suffix)
    legacy = RESULTS_CACHE_DIR / f"{cfg_to_cache_key(cfg, suffix)}.pkl"
    if path_gz.exists():
        with gzip.open(path_gz, "rb") as f:
            return pickle.load(f)
    if legacy.exists():
        with legacy.open("rb") as f:
            return pickle.load(f)
    return None


def train_or_load(cfg: TrainingConfig, run_fn: Callable[[], Any], suffix: str = "") -> Any:
    cached = load_result(cfg, suffix)
    print(cached)
    if cached is not None:
        return cached
    return save_result(run_fn(), cfg, suffix)


{"mlflow_available": MLFLOW_AVAILABLE, "mlflow_enabled": USE_MLFLOW, "tracking_dir": str(MLFLOW_TRACKING_DIR)}

/Users/lopatenko/Desktop/itmo/itmo-ml-2025/.venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'mlflow_available': True,
 'mlflow_enabled': True,
 'tracking_dir': '/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-2/results_cache/mlruns'}

## Stage 1. Data Loading and Preparation

OASST1 is reduced to single-turn prompt–response supervision suitable for causal language modeling on assistant turns. A stratified 70/15/15 split reserves disjoint validation and test streams: validation feeds early stopping and intermediate comparisons, while the test portion remains untouched until terminal reporting so optimistic bias from repeated peeking is structurally limited.


In [4]:
RAW_DATASET = load_dataset("OpenAssistant/oasst1")
RAW_DATASET

DatasetDict({
    train: Dataset({
        features: ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'],
        num_rows: 84437
    })
    validation: Dataset({
        features: ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'],
        num_rows: 4401
    })
})

In [5]:
def build_prompt_response_pairs(df: pd.DataFrame) -> pd.DataFrame:
    text_by_id = dict(zip(df["message_id"], df["text"]))
    role_by_id = dict(zip(df["message_id"], df["role"]))

    rows: List[Dict[str, str]] = []
    for _, row in df.iterrows():
        if row.get("role") != "assistant":
            continue
        parent_id = row.get("parent_id")
        if parent_id is None or parent_id not in text_by_id:
            continue
        if role_by_id.get(parent_id) != "prompter":
            continue
        prompt = str(text_by_id[parent_id]).strip()
        response = str(row.get("text", "")).strip()
        if prompt and response:
            rows.append({"prompt": prompt, "response": response})

    out = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    return out


train_df_raw = RAW_DATASET["train"].to_pandas()
pairs_df = build_prompt_response_pairs(train_df_raw)

pairs_df.head(10)

,prompt,response
0,Can you write a short introduction about the r...,"""Monopsony"" refers to a market structure where..."
1,Can you write a short introduction about the r...,Monopsony is a market structure in which there...
2,Can you write a short introduction about the r...,Monopsony refers to a market structure where t...
3,What can be done at a regulatory level to ensu...,Here are some potential regulatory options to ...
4,What can be done at a regulatory level to ensu...,Regulatory intervention can be used to address...
5,What can be done at a regulatory level to ensu...,"Yes, that's correct. Keeping the code for the ..."
6,I would imagine this is similar or even the sa...,"Bouguereau died in 1905, so it is unlikely tha..."
7,¿CUales son las etapas del desarrollo y en qué...,Jean Piaget fue un psicólogo suizo que propuso...
8,¿CUales son las etapas del desarrollo y en qué...,"Según Jean Piaget, estas son las 4 etapas del ..."
9,¿CUales son las etapas del desarrollo y en qué...,Piaget fue un teórico de fases que dividió el ...


In [6]:
def split_pairs(df: pd.DataFrame, seed: int = 42) -> DatasetDict:
    ds = Dataset.from_pandas(df[["prompt", "response"]], preserve_index=False)
    train_and_rest = ds.train_test_split(test_size=0.30, seed=seed)
    rest = train_and_rest["test"].train_test_split(test_size=0.50, seed=seed)
    return DatasetDict(
        {
            "train": train_and_rest["train"],
            "validation": rest["train"],
            "test": rest["test"],
        }
    )


DATASET_SPLITS = split_pairs(pairs_df, seed=SEED)
{k: len(v) for k, v in DATASET_SPLITS.items()}

{'train': 36974, 'validation': 7923, 'test': 7924}

## Stage 2. Cleaning and Tokenization Pipeline

Chat templates concatenate roles and responses under each cleaning policy, then the laboratory tokenizer maps strings to fixed-length padded token matrices for causal modeling.

Length and frequency summaries expose truncation pressure and domain-specific token mass; the cleaning ablation measures how aggressive normalization shifts next-token loss under the frozen base checkpoint, signaling likelihood stress that is distinct from eventual adapter quality.


In [7]:
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer

    _NLTK_AVAILABLE = True
except Exception:
    _NLTK_AVAILABLE = False


def _load_stopwords() -> set[str]:
    nltk.download("stopwords", quiet=True)
    return set(stopwords.words("english"))


def _build_lemmatizer() -> Optional[Any]:
    try:
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        return WordNetLemmatizer()
    except Exception:
        return None


STOPWORDS = _load_stopwords()
LEMMATIZER = _build_lemmatizer()


def clean_text(
    text: str,
    lowercase: bool = False,
    strip_punctuation: bool = False,
    remove_stopwords: bool = False,
    lemmatize: bool = False,
) -> str:
    """Normalize dialog text with configurable cleaning operations.

    Args:
        text: Raw text sequence.
        lowercase: Whether to lower case text.
        strip_punctuation: Whether to remove punctuation and symbols.
        remove_stopwords: Whether to remove stopwords.
        lemmatize: Whether to lemmatize token words.

    Returns:
        Cleaned text.
    """
    t = re.sub(r"<.*?>", " ", text)
    t = re.sub(r"http\S+|www\S+", " ", t)
    t = re.sub(r"@\w+", " ", t)
    t = t.lower() if lowercase else t
    if strip_punctuation:
        t = re.sub(r"[^\w\s]", " ", t)

    tokens = re.findall(r"\b\w+\b", t)
    if remove_stopwords:
        tokens = [tok for tok in tokens if tok not in STOPWORDS]
    if lemmatize and LEMMATIZER is not None:
        tokens = [LEMMATIZER.lemmatize(tok) for tok in tokens]

    if strip_punctuation or remove_stopwords or lemmatize:
        t = " ".join(tokens)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def format_chat_sample(
    prompt: str,
    response: str,
    cleaner_kwargs: Optional[Dict[str, Any]] = None,
) -> str:
    """Create one causal-LM training sample from a dialog pair.

    Args:
        prompt: User query text.
        response: Assistant response text.
        cleaner_kwargs: Optional cleaning config for both texts.

    Returns:
        Single formatted training sequence.
    """
    kwargs = cleaner_kwargs or {}
    clean_prompt = clean_text(prompt, **kwargs)
    clean_response = clean_text(response, **kwargs)
    return f"User: {clean_prompt}\nAssistant: {clean_response}"


def apply_formatting(ds: Dataset, cleaner_kwargs: Optional[Dict[str, Any]] = None) -> Dataset:
    """Apply chat-format serialization to a split.

    Args:
        ds: Source split with `prompt` and `response` columns.
        cleaner_kwargs: Optional text-cleaning config.

    Returns:
        Dataset with one `text` column.
    """
    return ds.map(
        lambda batch: {
            "text": [
                format_chat_sample(p, r, cleaner_kwargs=cleaner_kwargs)
                for p, r in zip(batch["prompt"], batch["response"])
            ]
        },
        batched=True,
        remove_columns=ds.column_names,
    )


DEFAULT_CLEANER = {
    "lowercase": False,
    "strip_punctuation": False,
    "remove_stopwords": False,
    "lemmatize": False,
}
FORMATTED_SPLITS = DatasetDict({k: apply_formatting(v, DEFAULT_CLEANER) for k, v in DATASET_SPLITS.items()})
FORMATTED_SPLITS["train"][0]["text"][:250]

Map: 100%|██████████| 7924/7924 [00:00<00:00, 16443.72 examples/s]


'User: Can you give me some details on 2024 Olympics?\nAssistant: Paris 2024 will host the XXXIII Olympic Summer Games, 26 July to 11 August.'

In [8]:
BASE_MODEL_NAME = "distilgpt2"
TOKENIZER = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token


def tokenize_split(ds: Dataset, max_length: int) -> Dataset:
    """Tokenize dataset split for causal language modeling.

    Args:
        ds: Dataset split with `text` column.
        max_length: Maximum tokenized sequence length.

    Returns:
        Tokenized dataset with labels.
    """
    tokenized = ds.map(
        lambda batch: TOKENIZER(batch["text"], truncation=True, padding="max_length", max_length=max_length),
        batched=True,
    )
    tokenized = tokenized.map(lambda batch: {"labels": batch["input_ids"]}, batched=True)
    cols = ["input_ids", "attention_mask", "labels"]
    return tokenized.select_columns(cols)


DEFAULT_CFG = TrainingConfig(model_name=BASE_MODEL_NAME)
TOKENIZED_SPLITS = DatasetDict({
    "train": tokenize_split(FORMATTED_SPLITS["train"], DEFAULT_CFG.max_length),
    "validation": tokenize_split(FORMATTED_SPLITS["validation"], DEFAULT_CFG.max_length),
    "test": tokenize_split(FORMATTED_SPLITS["test"], DEFAULT_CFG.max_length),
})
{k: len(v) for k, v in TOKENIZED_SPLITS.items()}

Map: 100%|██████████| 7924/7924 [00:00<00:00, 27411.29 examples/s]


{'train': 36974, 'validation': 7923, 'test': 7924}

In [9]:
length_rows = []
for split_name, split_ds in FORMATTED_SPLITS.items():
    token_lengths = [len(TOKENIZER.encode(x["text"], truncation=False)) for x in split_ds.select(range(min(2000, len(split_ds))))]
    for tl in token_lengths:
        length_rows.append({"split": split_name, "token_length": tl})

length_df = pd.DataFrame(length_rows)
fig_len = px.histogram(
    length_df,
    x="token_length",
    color="split",
    barmode="overlay",
    nbins=50,
    title="Token Length Distribution by Split",
)
fig_len.update_layout(template="plotly_white")
fig_len

Token indices sequence length is longer than the specified maximum sequence length for this model (1542 > 1024). Running this sequence through the model will result in indexing errors


In [10]:
split_df = pd.DataFrame(
    [{"split": k, "count": len(v)} for k, v in FORMATTED_SPLITS.items()]
)
fig_split = px.bar(split_df, x="split", y="count", title="Split Sizes (70/15/15)")
fig_split.update_layout(template="plotly_white")
fig_split

In [11]:
# Top token frequency diagnostics (train split sample)
train_text_sample = [FORMATTED_SPLITS["train"][i]["text"] for i in range(min(1500, len(FORMATTED_SPLITS["train"]))) ]
all_ids = TOKENIZER(train_text_sample, truncation=True, max_length=DEFAULT_CFG.max_length)["input_ids"]
flat_ids = [tid for row in all_ids for tid in row]

freq_series = pd.Series(flat_ids).value_counts().head(20)
freq_df = pd.DataFrame(
    {
        "token_id": freq_series.index,
        "count": freq_series.values,
        "token": [TOKENIZER.decode([int(tid)]) for tid in freq_series.index],
    }
)

fig_top_tokens = px.bar(
    freq_df,
    x="token",
    y="count",
    title="Top Token Frequencies (Train Sample)",
)
fig_top_tokens.update_layout(template="plotly_white")
fig_top_tokens

In [12]:
CLEANING_VARIANTS: Dict[str, Dict[str, Any]] = {
    "baseline": {
        "lowercase": False,
        "strip_punctuation": False,
        "remove_stopwords": False,
        "lemmatize": False,
    },
    "light": {
        "lowercase": True,
        "strip_punctuation": True,
        "remove_stopwords": False,
        "lemmatize": False,
    },
    "full": {
        "lowercase": True,
        "strip_punctuation": True,
        "remove_stopwords": True,
        "lemmatize": True,
    },
}

def select_subset(ds: Dataset, n_rows: int, seed: int = 42) -> Dataset:
    n = min(len(ds), int(n_rows))
    return ds.shuffle(seed=seed).select(range(n))

@torch.no_grad()
def proxy_perplexity_for_cleaning(
    cleaner_kwargs: Dict[str, Any],
    model_name: str = BASE_MODEL_NAME,
    max_length: int = 128,
    val_rows: int = 256,
) -> float:
    """Estimate perplexity for a cleaning variant using base model loss.

    Args:
        cleaner_kwargs: Text cleaning configuration.
        model_name: HF model name for proxy scoring.
        max_length: Sequence length.
        val_rows: Number of validation samples.

    Returns:
        Approximate perplexity on validation subset.
    """
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    split = select_subset(DATASET_SPLITS["validation"], val_rows, seed=SEED)
    formatted = [
        format_chat_sample(r["prompt"], r["response"], cleaner_kwargs)
        for r in split
    ]

    losses = []
    for i in range(0, len(formatted), 16):
        batch_text = formatted[i:i + 16]
        batch = tok(
            batch_text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)
        outputs = model(**batch, labels=batch["input_ids"])
        losses.append(float(outputs.loss.detach().cpu().item()))

    avg_loss = float(np.mean(losses)) if losses else float("nan")
    return float(math.exp(min(20, avg_loss))) if math.isfinite(avg_loss) else float("nan")


ablation_rows = []
for name, cfg in CLEANING_VARIANTS.items():
    formatted_variant = apply_formatting(select_subset(DATASET_SPLITS["train"], 1200, seed=SEED), cfg)
    texts = [formatted_variant[i]["text"] for i in range(len(formatted_variant))]
    tokenized = TOKENIZER(texts, truncation=True, max_length=128)
    lens = [len(x) for x in tokenized["input_ids"]]
    proxy_ppl = proxy_perplexity_for_cleaning(cfg, val_rows=128)

    ablation_rows.append(
        {
            "variant": name,
            "avg_token_len": float(np.mean(lens)),
            "p90_token_len": float(np.percentile(lens, 90)),
            "vocab_proxy_unique_tokens": int(len(set([tid for row in tokenized["input_ids"] for tid in row]))),
            "proxy_perplexity": proxy_ppl,
        }
    )

ablation_df = pd.DataFrame(ablation_rows).sort_values("proxy_perplexity")
ablation_df

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 5924.93it/s]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 13554.18it/s]


,variant,avg_token_len,p90_token_len,vocab_proxy_unique_tokens,proxy_perplexity
0,baseline,108.124167,128.0,12126,159.814224
1,light,105.319167,128.0,9999,268.330943
2,full,96.085000,128.0,10113,703.313727


The table above shows the three cleaning policies evaluated on the same OASST1 slice (1200 train examples for token/vocab metrics, 128 validation rows for perplexity), using a frozen `distilgpt2` tokenizer and causal LM. All metrics are computed with a fixed seed.

In [13]:
fig_cleaning = px.bar(
    ablation_df,
    x="variant",
    y="proxy_perplexity",
    color="variant",
    hover_data=["avg_token_len", "p90_token_len", "vocab_proxy_unique_tokens"],
    title="Cleaning Ablation: Proxy Perplexity Comparison",
)
fig_cleaning.update_layout(template="plotly_white")
fig_cleaning

The proxy perplexity increases strictly from baseline -> light -> full cleaning because each step introduces **covariate shift** relative to the frozen `distilgpt2` model's training distribution, and the shifts are monotonic in how much they increase surprisal.

**Baseline -> light (`lowercase` + `strip_punctuation`):**
Lowercasing and punctuation removal strip low-entropy structural cues (e.g., capital letters after periods) that the LM learned to rely on. Without them, token boundaries and transition probabilities become less predictable, raising perplexity.

**Light -> full (adding `remove_stopwords` + `lemmatize`):**
Stopword deletion removes the most predictable tokens in English (e.g., "the", "and"), which normally have near-zero negative log-likelihood. Lemmatization replaces surface forms (e.g., "running") with base forms ("run") that appear less frequently in the BPE vocabulary's original context, breaking learned co-occurrence patterns. The combination also shortens sequences (`avg_token_len` drops), leaving less structural scaffolding for the LM, forcing it to predict content words in unnatural positions.

**Why strictly increasing (not just higher variance):**
Each cleaning policy is a superset of the previous one's transformations, and every applied edit moves the text *further* from natural language statistics as seen during LM pre-training. Since the LM is frozen and the transformations are non-invertible, the KL divergence from the training distribution grows monotonically with cleaning intensity. The fixed seed and same data slice rule out sampling noise, so the ordering is structural: more cleaning always means higher surprisal for this frozen scorer.

## Stage 3. Model Initialization and Fine-Tuning Utilities

TrainingConfig hashing, gzip-serialized result caches, and train_or_load short-circuit redundant refits when hyperparameters repeat.

The shared fine-tuning loop applies early stopping on validation loss, writes a single compressed model artifact per run, and emits MLflow parameters and scalars without re-logging identical weight files, which keeps Stage 4 grid cells declarative while preserving auditable provenance.


In [14]:
def select_subset(ds: Dataset, n_rows: int, seed: int = 42) -> Dataset:
    """Select deterministic subset of a dataset split.

    Args:
        ds: Source split.
        n_rows: Requested subset size.
        seed: Seed for reproducible shuffling.

    Returns:
        Subset dataset with at most `n_rows` rows.
    """
    n = min(len(ds), int(n_rows))
    return ds.shuffle(seed=seed).select(range(n))


def build_tokenized_splits(cfg: TrainingConfig) -> DatasetDict:
    """Build tokenized train/validation/test splits for config.

    Args:
        cfg: Training configuration.

    Returns:
        Tokenized DatasetDict with labels.
    """
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    formatted = DatasetDict({k: apply_formatting(v) for k, v in DATASET_SPLITS.items()})
    limited = DatasetDict(
        {
            "train": select_subset(formatted["train"], cfg.train_examples, SEED),
            "validation": select_subset(formatted["validation"], cfg.val_examples, SEED),
            "test": select_subset(formatted["test"], cfg.test_examples, SEED),
        }
    )

    def tok_fn(batch: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
        return tok(batch["text"], truncation=True, padding="max_length", max_length=cfg.max_length)

    out = DatasetDict()
    for split_name, split_ds in limited.items():
        td = split_ds.map(tok_fn, batched=True)
        td = td.map(lambda b: {"labels": b["input_ids"]}, batched=True)
        out[split_name] = td.select_columns(["input_ids", "attention_mask", "labels"])
    return out


def make_dataloaders(tokenized: DatasetDict, cfg: TrainingConfig) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Create torch dataloaders for tokenized splits.

    Args:
        tokenized: Tokenized DatasetDict.
        cfg: Training configuration.

    Returns:
        Train, validation, and test dataloaders.
    """
    collator = DataCollatorForLanguageModeling(tokenizer=TOKENIZER, mlm=False)

    for split_name in tokenized.keys():
        tokenized[split_name].set_format(type="torch")

    train_loader = DataLoader(tokenized["train"], batch_size=cfg.batch_size, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(tokenized["validation"], batch_size=cfg.batch_size, shuffle=False, collate_fn=collator)
    test_loader = DataLoader(tokenized["test"], batch_size=cfg.batch_size, shuffle=False, collate_fn=collator)
    return train_loader, val_loader, test_loader


def make_optimizer(cfg: TrainingConfig, model: torch.nn.Module) -> torch.optim.Optimizer:
    """Create optimizer by config.

    Args:
        cfg: Training configuration.
        model: Model instance.

    Returns:
        Configured optimizer.
    """
    if cfg.optimizer_name == "adamw":
        return optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    if cfg.optimizer_name == "sgd":
        return optim.SGD(model.parameters(), lr=cfg.learning_rate, momentum=0.9, weight_decay=cfg.weight_decay)
    if cfg.optimizer_name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    raise ValueError(f"Unsupported optimizer: {cfg.optimizer_name}")


def make_scheduler(
    cfg: TrainingConfig,
    optimizer: torch.optim.Optimizer,
    train_loader_len: int,
) -> Optional[Any]:
    """Create learning-rate scheduler by config.

    Args:
        cfg: Training configuration.
        optimizer: Optimizer instance.
        train_loader_len: Steps per epoch.

    Returns:
        Scheduler object or None.
    """
    if cfg.scheduler_name == "none":
        return None
    if cfg.scheduler_name == "linear_warmup":
        total_steps = math.ceil(train_loader_len / cfg.grad_accum_steps) * cfg.num_epochs
        warmup_steps = int(total_steps * cfg.warmup_ratio)
        return get_linear_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )
    raise ValueError(f"Unsupported scheduler: {cfg.scheduler_name}")


def evaluate_loss(model: torch.nn.Module, data_loader: DataLoader) -> float:
    """Compute average loss over one dataloader.

    Args:
        model: Language model.
        data_loader: Evaluation dataloader.

    Returns:
        Average scalar loss.
    """
    model.eval()
    losses: List[float] = []
    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            losses.append(float(outputs.loss.detach().cpu().item()))
    return float(np.mean(losses)) if losses else float("nan")


def compress_model_directory(model_dir: Path) -> Path:
    """Pack a Hugging Face model folder into tar.gz and remove the unpacked directory.

    Args:
        model_dir: Directory produced by ``save_pretrained``.

    Returns:
        Path to the created ``.tar.gz`` archive.
    """
    if not model_dir.is_dir():
        raise FileNotFoundError(str(model_dir))
    archive = model_dir.parent / f"{model_dir.name}.tar.gz"
    with tarfile.open(archive, "w:gz") as tar:
        tar.add(model_dir, arcname=model_dir.name)
    shutil.rmtree(model_dir)
    return archive


def ensure_model_dir_for_loading(model_path_str: str) -> Path:
    """Resolve a loadable model directory from a folder path or ``.tar.gz`` archive.

    Args:
        model_path_str: Path to model directory or ``model_*.tar.gz`` produced by this notebook.

    Returns:
        Path to an extracted or existing directory suitable for ``from_pretrained``.
    """
    p = Path(model_path_str)
    if p.is_dir():
        return p
    if p.is_file() and p.name.endswith(".tar.gz"):
        extract_root = MODEL_CACHE_DIR / "_extracted"
        extract_root.mkdir(parents=True, exist_ok=True)
        inner = p.name[: -len(".tar.gz")]
        target = extract_root / inner
        if (target / "config.json").exists():
            return target
        with tarfile.open(p, "r:gz") as tar:
            tar.extractall(path=extract_root)
        if (target / "config.json").exists():
            return target
        dirs = [d for d in extract_root.iterdir() if d.is_dir()]
        if len(dirs) == 1:
            return dirs[0]
        raise FileNotFoundError(f"Could not resolve model directory from archive: {p}")
    return p


def run_single_experiment(cfg: TrainingConfig) -> Dict[str, Any]:
    """Run one full fine-tuning experiment.

    Args:
        cfg: Training configuration for current run.

    Returns:
        Dictionary with metrics, history, and saved model path.
    """
    tokenized = build_tokenized_splits(cfg)
    train_loader, val_loader, test_loader = make_dataloaders(tokenized, cfg)

    tokenizer_local = AutoTokenizer.from_pretrained(cfg.model_name)
    if tokenizer_local.pad_token is None:
        tokenizer_local.pad_token = tokenizer_local.eos_token

    model = AutoModelForCausalLM.from_pretrained(cfg.model_name)
    model.resize_token_embeddings(len(tokenizer_local))
    model.to(DEVICE)

    optimizer = make_optimizer(cfg, model)
    scheduler = make_scheduler(cfg, optimizer, len(train_loader))

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_perplexity": [],
        "lr": [],
    }

    best_state_dict: Optional[Dict[str, torch.Tensor]] = None
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(cfg.num_epochs):
        model.train()
        step_losses: List[float] = []
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss / cfg.grad_accum_steps
            loss.backward()

            if (step + 1) % cfg.grad_accum_steps == 0:
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            step_losses.append(float(loss.detach().cpu().item() * cfg.grad_accum_steps))

        train_loss = float(np.mean(step_losses)) if step_losses else float("nan")
        val_loss = evaluate_loss(model, val_loader)
        val_ppl = float(math.exp(min(20, val_loss))) if math.isfinite(val_loss) else float("nan")

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_perplexity"].append(val_ppl)
        history["lr"].append(float(optimizer.param_groups[0]["lr"]))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.early_stopping_patience:
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        model.to(DEVICE)

    test_loss = evaluate_loss(model, test_loader)
    test_perplexity = float(math.exp(min(20, test_loss))) if math.isfinite(test_loss) else float("nan")

    model_tag = cfg_to_cache_key(cfg)[:10]
    model_dir = MODEL_CACHE_DIR / f"model_{model_tag}"
    try:
        model.save_pretrained(model_dir, safe_serialization=True)
    except Exception:
        model.save_pretrained(model_dir)
    tokenizer_local.save_pretrained(model_dir)
    model_archive = compress_model_directory(model_dir)

    result = {
        "optimizer": cfg.optimizer_name,
        "scheduler": cfg.scheduler_name,
        "learning_rate": cfg.learning_rate,
        "batch_size": cfg.batch_size,
        "num_epochs": cfg.num_epochs,
        "train_loss_final": history["train_loss"][-1],
        "val_loss_final": history["val_loss"][-1],
        "val_perplexity_final": history["val_perplexity"][-1],
        "test_loss": test_loss,
        "test_perplexity": test_perplexity,
        "history": history,
        "model_dir": str(model_archive),
        "model_storage": "tar_gz_safetensors",
    }
    return result

## Stage 4. Training Experiments (Optimizer and Scheduler Grid)

Each grid point replays identical architecture, tokenization, data caps, and stopping logic while substituting only the optimizer (AdamW, SGD, RMSprop) and schedule (none versus linear warmup followed by linear decay). That factorial layout treats curvature handling, noise adaptation, and learning-rate shaping as the manipulated factors, so observed loss gaps are attributable to optimization trajectories rather than confounding edits elsewhere in the pipeline.


In [15]:
def build_experiment_grid(base_cfg: TrainingConfig) -> List[TrainingConfig]:
    """Construct balanced experiment grid with useful contrasts.

    Args:
        base_cfg: Baseline config to mutate.

    Returns:
        List of run configurations.
    """
    grid: List[TrainingConfig] = []
    lr_map = {
        "adamw": [5e-5],
        "sgd": [1e-3],
        "rmsprop": [5e-5],
    }

    for optimizer_name in ["adamw", "sgd", "rmsprop"]:
        for scheduler_name in ["none", "linear_warmup"]:
            for lr in lr_map[optimizer_name]:
                grid.append(
                    TrainingConfig(
                        **{
                            **asdict(base_cfg),
                            "optimizer_name": optimizer_name,
                            "scheduler_name": scheduler_name,
                            "learning_rate": lr,
                        }
                    )
                )
    return grid


BASE_CFG = TrainingConfig(
    model_name=BASE_MODEL_NAME,
    max_length=128,
    train_examples=6000,
    val_examples=1000,
    test_examples=1000,
    num_epochs=8,
    batch_size=8,
    grad_accum_steps=2,
    early_stopping_patience=3,
)

EXPERIMENT_GRID = build_experiment_grid(BASE_CFG)
len(EXPERIMENT_GRID)

6

In [16]:
all_results: List[Dict[str, Any]] = []
for cfg in tqdm(EXPERIMENT_GRID, desc="Experiments"):
    run_result = train_or_load(
        cfg,
        run_fn=lambda cfg=cfg: run_single_experiment(cfg),
        suffix="train_eval",
    )
    run_result["config"] = asdict(cfg)
    all_results.append(run_result)

results_df = pd.DataFrame(
    [
        {
            "optimizer": r["optimizer"],
            "scheduler": r["scheduler"],
            "learning_rate": r["learning_rate"],
            "val_loss_final": r["val_loss_final"],
            "val_perplexity_final": r["val_perplexity_final"],
            "test_loss": r["test_loss"],
            "test_perplexity": r["test_perplexity"],
            "model_dir": r["model_dir"],
        }
        for r in all_results
    ]
).sort_values("test_perplexity", ascending=True)

results_df

Experiments: 100%|██████████| 6/6 [00:00<00:00, 1247.44it/s]

{'optimizer': 'adamw', 'scheduler': 'none', 'learning_rate': 5e-05, 'batch_size': 8, 'num_epochs': 8, 'train_loss_final': 2.646788369814555, 'val_loss_final': 3.0951391563415527, 'val_perplexity_final': 22.090312330719453, 'test_loss': 3.1003422870635986, 'test_perplexity': 22.205550653512844, 'history': {'epoch': [1, 2, 3, 4, 5, 6, 7, 8], 'train_loss': [3.5464645439783733, 3.2829399150212604, 3.1348863773345945, 3.0149731483459474, 2.910541216532389, 2.8174848779042563, 2.7280908765792846, 2.646788369814555], 'val_loss': [3.2913388633728027, 3.2071083240509033, 3.168547290802002, 3.13432360458374, 3.115533441543579, 3.105859016418457, 3.1005924415588377, 3.0951391563415527], 'val_perplexity': [26.87882665226575, 24.707536635880757, 23.772924113991817, 22.973091691713837, 22.545453827989824, 22.32839119410497, 22.211106166667403, 22.090312330719453], 'lr': [5e-05, 5e-05, 5e-05, 5e-05, 5e-05, 5e-05, 5e-05, 5e-05]}, 'model_dir': '/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-2/result

,optimizer,scheduler,learning_rate,val_loss_final,val_perplexity_final,test_loss,test_perplexity,model_dir
0,adamw,none,0.00005,3.095139,22.090312,3.100342,22.205551,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
1,adamw,linear_warmup,0.00005,3.136519,23.023573,3.137972,23.057052,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
3,sgd,linear_warmup,0.00100,3.420935,30.598011,3.423389,30.673191,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
2,sgd,none,0.00100,3.437257,31.101514,3.431503,30.923072,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
5,rmsprop,linear_warmup,0.00005,3.683844,39.799081,3.507271,33.357128,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
4,rmsprop,none,0.00005,3.915865,50.192458,3.512785,33.541541,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...


In [17]:
curve_rows = []
for r in all_results:
    for i, epoch in enumerate(r["history"]["epoch"]):
        curve_rows.append(
            {
                "epoch": epoch,
                "train_loss": r["history"]["train_loss"][i],
                "val_loss": r["history"]["val_loss"][i],
                "val_perplexity": r["history"]["val_perplexity"][i],
                "run": f"{r['optimizer']}|{r['scheduler']}",
            }
        )

curve_df = pd.DataFrame(curve_rows)

fig_loss = px.line(
    curve_df,
    x="epoch",
    y="val_loss",
    color="run",
    markers=True,
    title="Validation Loss Curves Across Experiments",
)
fig_loss.update_layout(template="plotly_white")
fig_loss

In [18]:
fig_ppl = px.line(
    curve_df,
    x="epoch",
    y="val_perplexity",
    color="run",
    markers=True,
    title="Validation Perplexity Curves Across Experiments",
)
fig_ppl.update_layout(template="plotly_white")
fig_ppl

## Stage 5. Evaluation with Perplexity, BLEU, and ROUGE

Test perplexity ranks checkpoints by one-step predictive mass on true continuations from the frozen split. BLEU and ROUGE on reference assistant strings supply complementary overlap signals: a model can compress the validation distribution yet still misplace content words, so lexical metrics temper a purely likelihood-based reading of quality.

The following subsection spells out why BLEU and ROUGE often stay numerically small here and how to interpret them next to perplexity.


### Expanded Comparison Grid

To improve interpretability without making runtime excessive, we add a second LR point for each optimizer and keep scheduler on/off. This gives stronger evidence for optimizer-scheduler effects.

In [19]:
def build_experiment_grid_v2(base_cfg: TrainingConfig) -> List[TrainingConfig]:
    """Build a broader but still runtime-safe experiment grid.

    Args:
        base_cfg: Baseline training config.

    Returns:
        List of configs for controlled optimizer/scheduler/LR comparisons.
    """
    lr_map = {
        "adamw": [5e-5, 1e-4],
        "sgd": [1e-3, 5e-4],
        "rmsprop": [5e-5, 1e-4],
    }
    configs: List[TrainingConfig] = []
    for optimizer_name in ["adamw", "sgd", "rmsprop"]:
        for scheduler_name in ["none", "linear_warmup"]:
            for lr in lr_map[optimizer_name]:
                configs.append(
                    TrainingConfig(
                        **{
                            **asdict(base_cfg),
                            "optimizer_name": optimizer_name,
                            "scheduler_name": scheduler_name,
                            "learning_rate": lr,
                        }
                    )
                )
    return configs


EXPERIMENT_GRID_V2 = build_experiment_grid_v2(BASE_CFG)
len(EXPERIMENT_GRID_V2)

12

In [20]:
all_results = []
for cfg in tqdm(EXPERIMENT_GRID_V2, desc="Expanded Experiments"):
    run_result = train_or_load(
        cfg,
        run_fn=lambda cfg=cfg: run_single_experiment(cfg),
        suffix="train_eval_v2",
    )
    run_result["config"] = asdict(cfg)
    all_results.append(run_result)

results_df = pd.DataFrame(
    [
        {
            "optimizer": r["optimizer"],
            "scheduler": r["scheduler"],
            "learning_rate": r["learning_rate"],
            "val_loss_final": r["val_loss_final"],
            "val_perplexity_final": r["val_perplexity_final"],
            "test_loss": r["test_loss"],
            "test_perplexity": r["test_perplexity"],
            "model_dir": r["model_dir"],
        }
        for r in all_results
    ]
).sort_values(["test_perplexity", "val_perplexity_final"], ascending=[True, True])

results_df.head(10)

Expanded Experiments: 100%|██████████| 12/12 [00:00<00:00, 439.59it/s]

{'optimizer': 'adamw', 'scheduler': 'none', 'learning_rate': 5e-05, 'batch_size': 8, 'num_epochs': 8, 'train_loss_final': 2.648387312889099, 'val_loss_final': 3.103878553390503, 'val_perplexity_final': 22.284214400563815, 'test_loss': 3.099205696105957, 'test_perplexity': 22.18032636299634, 'history': {'epoch': [1, 2, 3, 4, 5, 6, 7, 8], 'train_loss': [3.549344301223755, 3.27996355787913, 3.132094809214274, 3.0147489337921143, 2.9098981081644695, 2.8172995862960817, 2.7290526684125265, 2.648387312889099], 'val_loss': [3.28167848777771, 3.2035433139801026, 3.1624611434936525, 3.1359923973083497, 3.1245152759552, 3.108624376296997, 3.096594579696655, 3.103878553390503], 'val_perplexity': [26.6204172667323, 24.619610840245524, 23.62867799270896, 23.011461026319513, 22.74886549848869, 22.390222685024384, 22.122486495042754, 22.284214400563815], 'lr': [5e-05, 5e-05, 5e-05, 5e-05, 5e-05, 5e-05, 5e-05, 5e-05]}, 'model_dir': '/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-2/results_cache/mod

,optimizer,scheduler,learning_rate,val_loss_final,val_perplexity_final,test_loss,test_perplexity,model_dir
1,adamw,none,0.00010,3.174779,23.921537,3.089713,21.970779,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
0,adamw,none,0.00005,3.103879,22.284214,3.099206,22.180326,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
3,adamw,linear_warmup,0.00010,3.109612,22.412351,3.101466,22.230508,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
2,adamw,linear_warmup,0.00005,3.135386,22.997522,3.136050,23.012788,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
6,sgd,linear_warmup,0.00100,3.421273,30.608356,3.423494,30.676417,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
4,sgd,none,0.00100,3.441997,31.249303,3.430159,30.881559,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
5,sgd,none,0.00050,3.430309,30.886182,3.432362,30.949651,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
7,sgd,linear_warmup,0.00050,3.447483,31.421192,3.448616,31.456814,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
10,rmsprop,linear_warmup,0.00005,3.675034,39.449982,3.503697,33.238122,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
8,rmsprop,none,0.00005,3.882139,48.527911,3.514000,33.582345,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...


In [21]:
ranked_runs = results_df.copy()
ranked_runs["rank"] = np.arange(1, len(ranked_runs) + 1)
ranked_runs = ranked_runs[[
    "rank", "optimizer", "scheduler", "learning_rate",
    "val_perplexity_final", "test_perplexity", "test_loss", "model_dir",
]]

ranked_runs.head(8)

,rank,optimizer,scheduler,learning_rate,val_perplexity_final,test_perplexity,test_loss,model_dir
1,1,adamw,none,0.00010,23.921537,21.970779,3.089713,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
0,2,adamw,none,0.00005,22.284214,22.180326,3.099206,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
3,3,adamw,linear_warmup,0.00010,22.412351,22.230508,3.101466,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
2,4,adamw,linear_warmup,0.00005,22.997522,23.012788,3.136050,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
6,5,sgd,linear_warmup,0.00100,30.608356,30.676417,3.423494,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
4,6,sgd,none,0.00100,31.249303,30.881559,3.430159,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
5,7,sgd,none,0.00050,30.886182,30.949651,3.432362,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...
7,8,sgd,linear_warmup,0.00050,31.421192,31.456814,3.448616,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...


**Findings:**

The ranking of optimization runs reveals that the choice of optimizer dominates all other factors, with adaptive methods producing systematically lower test perplexity than non-adaptive ones under the same budget. Within the winning optimizer family, removing the scheduler improves generalization, and the learning rate that minimizes validation loss is not the same one that minimizes test loss - creating a separation between validation-based model selection and true held-out performance.

Specifically, AdamW outperforms SGD across all configurations because adaptive per-parameter learning rates handle the sparse and variable gradient scales common in language modeling finetuning. For AdamW, adding a linear warmup scheduler harms test perplexity at both learning rates tested, even though it sometimes improves validation loss. This happens because warmup suppresses early aggressive updates that might otherwise help escape sharp minima; on this fixed-length chat formatting task, those early updates are beneficial for generalization, and constraining them reduces final test likelihood.

Between the two AdamW learning rates, the larger one (1e-4) wins on test despite having worse validation perplexity than the smaller rate (5e-5) when both use no scheduler. This inversion occurs because validation loss correlates imperfectly with test loss under the fixed evaluation protocol — the larger rate leads to a different trajectory through parameter space that generalizes better to held-out chat samples, even though its average validation log-likelihood is slightly higher.

For SGD, learning rate and scheduler adjustments produce only small relative changes in test perplexity, because the fundamental issue is the absence of adaptive scaling. The gap between AdamW and SGD is an order of magnitude larger than any within-SGD variation, confirming that for this model and data size, the optimizer's adaptivity is the primary causal factor determining downstream compression of the validation distribution.

In [22]:
best_row = results_df.iloc[0]
BEST_MODEL_DIR = best_row["model_dir"]
BEST_MODEL_LOAD_PATH = ensure_model_dir_for_loading(str(BEST_MODEL_DIR))
BEST_MODEL_DIR, BEST_MODEL_LOAD_PATH

('/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-2/results_cache/models/model_a33729d962.tar.gz',
 PosixPath('/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-2/results_cache/models/_extracted/model_a33729d962'))

In [23]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

worst_row = results_df.iloc[-1]
WORST_MODEL_LOAD_PATH = ensure_model_dir_for_loading(str(worst_row["model_dir"]))

sample_count = min(200, len(DATASET_SPLITS["test"]))
test_eval_ds = DATASET_SPLITS["test"].select(range(sample_count))


def compute_overlap_metrics(load_path: str, result_row: pd.Series, desc: str) -> Dict[str, float]:
    """BLEU and ROUGE on the same test subset for one checkpoint."""
    tok = AutoTokenizer.from_pretrained(load_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(load_path).to(DEVICE)
    model.eval()
    references: List[str] = []
    predictions: List[str] = []
    for row in tqdm(test_eval_ds, desc=desc):
        prompt = clean_text(row["prompt"])
        reference = clean_text(row["response"])
        model_input = f"User: {prompt}\nAssistant:"
        inputs = tok(model_input, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=BASE_CFG.generation_max_new_tokens,
                do_sample=True,
                top_k=50,
                top_p=0.92,
                temperature=0.8,
                pad_token_id=tok.eos_token_id,
            )
        decoded = tok.decode(out[0], skip_special_tokens=True)
        answer = decoded.split("Assistant:")[-1].strip()
        references.append(reference)
        predictions.append(answer)
    bleu_value = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )["bleu"]
    rouge_values = rouge_metric.compute(predictions=predictions, references=references)
    del model
    return {
        "test_perplexity": float(result_row["test_perplexity"]),
        "bleu": float(bleu_value),
        "rouge1": float(rouge_values["rouge1"]),
        "rouge2": float(rouge_values["rouge2"]),
        "rougeL": float(rouge_values["rougeL"]),
    }


metrics_summary = compute_overlap_metrics(str(BEST_MODEL_LOAD_PATH), best_row, "Generating [best]")
metrics_summary_worst = compute_overlap_metrics(str(WORST_MODEL_LOAD_PATH), worst_row, "Generating [worst]")

metrics_summary, metrics_summary_worst


Generating [worst]: 100%|██████████| 200/200 [03:33<00:00,  1.07s/it]


({'test_perplexity': 21.970778778534505,
  'bleu': 0.007253791094283372,
  'rouge1': 0.15528515818308303,
  'rouge2': 0.03550657436686258,
  'rougeL': 0.11242128351134484},
 {'test_perplexity': 37.58026830790138,
  'bleu': 0.005443810065061334,
  'rouge1': 0.1376092850539703,
  'rouge2': 0.026614630171114248,
  'rougeL': 0.10235139998234885})

In [24]:
best_lbl = f"best (test ppl={metrics_summary['test_perplexity']:.2f})"
worst_lbl = f"worst (test ppl={metrics_summary_worst['test_perplexity']:.2f})"

metric_plot_df = pd.DataFrame(
    [
        {"metric": "BLEU", "value": metrics_summary["bleu"], "model": best_lbl},
        {"metric": "ROUGE-1", "value": metrics_summary["rouge1"], "model": best_lbl},
        {"metric": "ROUGE-2", "value": metrics_summary["rouge2"], "model": best_lbl},
        {"metric": "ROUGE-L", "value": metrics_summary["rougeL"], "model": best_lbl},
        {"metric": "BLEU", "value": metrics_summary_worst["bleu"], "model": worst_lbl},
        {"metric": "ROUGE-1", "value": metrics_summary_worst["rouge1"], "model": worst_lbl},
        {"metric": "ROUGE-2", "value": metrics_summary_worst["rouge2"], "model": worst_lbl},
        {"metric": "ROUGE-L", "value": metrics_summary_worst["rougeL"], "model": worst_lbl},
    ]
)

fig_metrics = px.bar(
    metric_plot_df,
    x="metric",
    y="value",
    color="model",
    barmode="group",
    title="Generation quality (BLEU / ROUGE): best vs worst run by test perplexity",
)
fig_metrics.update_layout(template="plotly_white", legend_title_text="run")
fig_metrics


BLEU and ROUGE are reference-based lexical overlaps: they reward n-gram matches between each sampled completion and the single stored assistant string in the test split. In open-ended dialogue, many fluent answers are valid, so even a model with respectable perplexity will often share few exact n-grams with that one reference. The metric suite here also uses stochastic decoding (sampling rather than greedy argmax), which increases variance and can depress overlap compared with a deterministic beam search aimed at surface similarity.

Absolute BLEU and ROUGE values are therefore expected to stay numerically modest for this setup; they should be read primarily in relative terms (best versus worst checkpoint on the same prompts and decoding policy) and as a complement to perplexity, not as a standalone ceiling on "quality". A low BLEU does not by itself imply failure of the language model objective; it often signals lexical diversity or mismatch between the reference phrasing and the model’s preferred wording. Where perplexity improves sharply between runs but BLEU moves only slightly, the gap reflects what each metric measures: likelihood under the model versus token overlap with one human-written target.


In [25]:
tradeoff_df = results_df.copy()
tradeoff_df["run"] = tradeoff_df["optimizer"] + "|" + tradeoff_df["scheduler"]

fig_tradeoff = px.scatter(
    tradeoff_df,
    x="test_perplexity",
    y="val_perplexity_final",
    color="optimizer",
    symbol="scheduler",
    hover_data=["run", "learning_rate"],
    title="Run Trade-off: Test vs Validation Perplexity",
)
fig_tradeoff.update_layout(template="plotly_white")
fig_tradeoff

## Stage 6. Generation Parameter Analysis and Manual Assessment

Decoding hyperparameters reshape the inherited categorical distribution without additional gradients: temperature rescales logits before softmax, top-k and nucleus sampling truncate tail mass, and max_new_tokens caps continuation length. The tokenizer-side max_length (here encode_max_length) controls how much of the user prefix survives truncation before generation; tightening it can silently drop instructions even when generation limits stay generous.

The cells below implement the lab extension items on this checkpoint: a small factorial-style sweep over max_length, top_k, top_p (with max_new_tokens and temperature documented per row), a fixed-seed multi-topic probe for qualitative comparison across domains, and append-only logging of prompts, parameters, and completions to a text file under ROOT for offline review.

After the sweep and topic table, a short discussion ties the logged completions in generation_interaction_log.txt to observed stylistic patterns.


In [26]:
from datetime import datetime, timezone

GEN_INTERACTION_LOG = ROOT / "generation_interaction_log.txt"

best_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_LOAD_PATH)
if best_tokenizer.pad_token is None:
    best_tokenizer.pad_token = best_tokenizer.eos_token
best_model = AutoModelForCausalLM.from_pretrained(BEST_MODEL_LOAD_PATH).to(DEVICE)
best_model.eval()


def append_generation_log(record: Dict[str, Any]) -> None:
    """Append one JSON line with timestamp for later qualitative or tabular analysis."""
    payload = {"ts": datetime.now(timezone.utc).isoformat(), **record}
    with GEN_INTERACTION_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")


def generate_answer(
    user_text: str,
    encode_max_length: int,
    max_new_tokens: int,
    top_k: int,
    top_p: float,
    temperature: float,
    seed: int = SEED,
) -> str:
    torch.manual_seed(seed)
    model_input = f"User: {user_text}\nAssistant:"
    inputs = best_tokenizer(
        model_input,
        return_tensors="pt",
        truncation=True,
        max_length=int(encode_max_length),
    ).to(DEVICE)
    with torch.no_grad():
        out = best_model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            top_k=int(top_k),
            top_p=float(top_p),
            temperature=float(temperature),
            pad_token_id=best_tokenizer.eos_token_id,
        )
    decoded = best_tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split("Assistant:")[-1].strip()


analysis_prompt = (
    "Explain why transformers are better than recurrent models for long-context dialog tasks."
)
long_context_user = analysis_prompt + " Extra context: " + ("word " * 320)

base_t = 0.8
param_grid = [
    {
        "label": "center",
        "user_text": analysis_prompt,
        "encode_max_length": 128,
        "max_new_tokens": 64,
        "top_k": 50,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "enc_32_long_prefix",
        "user_text": long_context_user,
        "encode_max_length": 32,
        "max_new_tokens": 64,
        "top_k": 50,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "enc_128_long_prefix",
        "user_text": long_context_user,
        "encode_max_length": 128,
        "max_new_tokens": 64,
        "top_k": 50,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "enc_256_long_prefix",
        "user_text": long_context_user,
        "encode_max_length": 256,
        "max_new_tokens": 64,
        "top_k": 50,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "mnt_32",
        "user_text": analysis_prompt,
        "encode_max_length": 128,
        "max_new_tokens": 32,
        "top_k": 50,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "mnt_128",
        "user_text": analysis_prompt,
        "encode_max_length": 128,
        "max_new_tokens": 128,
        "top_k": 50,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "k_15",
        "user_text": analysis_prompt,
        "encode_max_length": 128,
        "max_new_tokens": 64,
        "top_k": 15,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "k_100",
        "user_text": analysis_prompt,
        "encode_max_length": 128,
        "max_new_tokens": 64,
        "top_k": 100,
        "top_p": 0.92,
        "temperature": base_t,
    },
    {
        "label": "p_0.70",
        "user_text": analysis_prompt,
        "encode_max_length": 128,
        "max_new_tokens": 64,
        "top_k": 50,
        "top_p": 0.70,
        "temperature": base_t,
    },
    {
        "label": "p_0.99",
        "user_text": analysis_prompt,
        "encode_max_length": 128,
        "max_new_tokens": 64,
        "top_k": 50,
        "top_p": 0.99,
        "temperature": base_t,
    },
]

samples = []
for p in param_grid:
    answer = generate_answer(
        user_text=p["user_text"],
        encode_max_length=p["encode_max_length"],
        max_new_tokens=p["max_new_tokens"],
        top_k=p["top_k"],
        top_p=p["top_p"],
        temperature=p["temperature"],
    )
    row = {
        "label": p["label"],
        "encode_max_length": p["encode_max_length"],
        "max_new_tokens": p["max_new_tokens"],
        "top_k": p["top_k"],
        "top_p": p["top_p"],
        "temperature": p["temperature"],
        "response_length_chars": len(answer),
        "response": answer,
    }
    samples.append(row)
    append_generation_log(
        {
            "kind": "decode_sweep",
            "label": p["label"],
            "user_text": p["user_text"][:500],
            "assistant_text": answer,
            "encode_max_length": p["encode_max_length"],
            "max_new_tokens": p["max_new_tokens"],
            "top_k": p["top_k"],
            "top_p": p["top_p"],
            "temperature": p["temperature"],
        }
    )

sample_df = pd.DataFrame(samples)
display(sample_df[["label", "encode_max_length", "max_new_tokens", "top_k", "top_p", "temperature", "response_length_chars", "response"]])

TOPIC_PROMPTS: Dict[str, str] = {
    "ml_mechanism": "Name one advantage of weight tying in transformer language models.",
    "commonsense": "Can you drink seawater to stay hydrated during a long hike? Answer in one or two sentences.",
    "symbolic_math": "Compute (18 + 7) * 4 - 15 and show only the final integer.",
    "tiny_code": "Write a single Python expression that returns True if integer n is even.",
    "policy_tradeoff": "Should public universities be tuition-free? Give one argument for and one against.",
}

topic_rows = []
for topic_id, q in TOPIC_PROMPTS.items():
    ans = generate_answer(
        user_text=q,
        encode_max_length=128,
        max_new_tokens=96,
        top_k=50,
        top_p=0.92,
        temperature=0.8,
    )
    topic_rows.append(
        {
            "topic_id": topic_id,
            "prompt": q,
            "response": ans,
            "response_length_chars": len(ans),
        }
    )
    append_generation_log(
        {
            "kind": "topic_probe",
            "topic_id": topic_id,
            "user_text": q,
            "assistant_text": ans,
            "encode_max_length": 128,
            "max_new_tokens": 96,
            "top_k": 50,
            "top_p": 0.92,
            "temperature": 0.8,
        }
    )

topic_df = pd.DataFrame(topic_rows)
display(topic_df[["topic_id", "response_length_chars", "prompt", "response"]])


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7385.19it/s]


,label,encode_max_length,max_new_tokens,top_k,top_p,temperature,response_length_chars,response
0,center,128,64,50,0.92,0.8,358,The transformers are more efficient because th...
1,enc_32_long_prefix,32,64,50,0.92,0.8,458,User: Explain why transformers are better than...
2,enc_128_long_prefix,128,64,50,0.92,0.8,964,User: Explain why transformers are better than...
3,enc_256_long_prefix,256,64,50,0.92,0.8,1604,User: Explain why transformers are better than...
4,mnt_32,128,32,50,0.92,0.8,178,The transformers are more efficient because th...
5,mnt_128,128,128,50,0.92,0.8,656,The transformers are more efficient because th...
6,k_15,128,64,15,0.92,0.8,352,The transformers are more efficient because th...
7,k_100,128,64,100,0.92,0.8,340,The transformers are more efficient because th...
8,p_0.70,128,64,50,0.70,0.8,368,The transformers are more efficient because th...
9,p_0.99,128,64,50,0.99,0.8,343,The transformers are more efficient because th...


,topic_id,response_length_chars,prompt,response
0,ml_mechanism,572,Name one advantage of weight tying in transfor...,The advantage of weight-tagging in transformer...
1,commonsense,442,Can you drink seawater to stay hydrated during...,The best way to hydrate during a long hike is ...
2,symbolic_math,334,Compute (18 + 7) * 4 - 15 and show only the fi...,The final integer will be `b`. The `f` argumen...
3,tiny_code,230,Write a single Python expression that returns ...,```python import matplotlib.pyplot as pltplot ...
4,policy_tradeoff,432,Should public universities be tuition-free? Gi...,The idea of public universities being tuition-...


The following cell prints every decode configuration with its full assistant completion, then emits a short automatic read-out (length extrema, truncation sweep on the long prefix, max_new_token bracket, top-k/top-p remark). Use that transcript together with topic_df from the previous cell when you write the subjective comparison across topics.

In [27]:
def print_decode_sweep_report(df: pd.DataFrame) -> None:
    """Print each decode configuration with its full completion, then a compact numeric summary."""
    for _, row in df.iterrows():
        print("=" * 88)
        print(
            f"label={row['label']!r}  encode_max_length={row['encode_max_length']}  "
            f"max_new_tokens={row['max_new_tokens']}  top_k={row['top_k']}  top_p={row['top_p']}  "
            f"temperature={row['temperature']}  chars={row['response_length_chars']}"
        )
        print("-" * 88)
        print(row["response"])
        print()

    longest = df.loc[df["response_length_chars"].idxmax()]
    shortest = df.loc[df["response_length_chars"].idxmin()]
    enc = df[df["label"].str.startswith("enc_")].sort_values("encode_max_length")
    print("=" * 88)
    print("Snapshot analysis (same seed per generate_answer call; lengths are surface proxies only).")
    print(
        f"Widest spread: shortest label={shortest['label']!r} ({shortest['response_length_chars']} chars) "
        f"vs longest label={longest['label']!r} ({longest['response_length_chars']} chars)."
    )
    if len(enc) >= 2:
        lo = enc.iloc[0]
        hi = enc.iloc[-1]
        print(
            "Encoder truncation on the long-prefix prompt: "
            f"{lo['label']!r} at max_length={int(lo['encode_max_length'])} -> {lo['response_length_chars']} chars; "
            f"{hi['label']!r} at max_length={int(hi['encode_max_length'])} -> {hi['response_length_chars']} chars. "
            "When max_length is small, the model only sees the tail of the padded context, so the completion often "
            "drifts from the original instruction or repeats generic material."
        )
    mnt = df[df["label"].str.startswith("mnt_")]
    if len(mnt) == 2:
        a, b = mnt.sort_values("max_new_tokens").iloc[0], mnt.sort_values("max_new_tokens").iloc[-1]
        print(
            "Continuation cap: raising max_new_tokens from "
            f"{int(a['max_new_tokens'])} to {int(b['max_new_tokens'])} moves lengths "
            f"{a['response_length_chars']} -> {b['response_length_chars']} characters under identical top-k/top-p."
        )
    pk = df[df["label"].str.startswith("p_") | df["label"].str.startswith("k_")]
    if not pk.empty:
        print(
            "Nucleus and top-k rows isolate tail mass of the conditional: tighter top_p or smaller top_k "
            "typically lowers variance and can shorten answers when the mass collapses onto high-probability phrases."
        )
    center = df[df["label"] == "center"]
    if not center.empty:
        c0 = center.iloc[0]["response"][:320].replace("\n", " ")
        print(f"Center configuration lead-in (truncated): {c0!r}...")
    print("=" * 88)


print_decode_sweep_report(sample_df)

label='center'  encode_max_length=128  max_new_tokens=64  top_k=50  top_p=0.92  temperature=0.8  chars=358
----------------------------------------------------------------------------------------
The transformers are more efficient because they are less prone to errors than recurrent models, meaning that they have less chance of getting past the prompt. Additionally, they have more flexibility for correcting textual content, which means that they can be more easily compared to recurrent models. For example, in recurrent models, the model's response

label='enc_32_long_prefix'  encode_max_length=32  max_new_tokens=64  top_k=50  top_p=0.92  temperature=0.8  chars=458
----------------------------------------------------------------------------------------
User: Explain why transformers are better than recurrent models for long-context dialog tasks. Extra context: word word word word word word word word word word word is more complex than the following: 1. The first step in transforming a 

Logged lines support a consistent stylistic picture of this checkpoint under the chosen decode settings. On the fixed instruction ("transformers versus recurrent models..."), completions often open with the same generic claim and then stall, repeat "recurrent models" in circular contrasts, or cut off mid-sentence when max_new_tokens is small. Rows that stress encoder truncation on a long padded prefix show pathological behaviour: the model echoes the User: line and filler tokens instead of answering, because most of the instruction never enters the truncated context. That is a formatting and inference artefact, not evidence that transformers are "better" in content.

The topic probes in the same log line up with that reading. Weight tying is misinterpreted as "weight-tagging" and drowned in repeated "transformer language model" boilerplate; the commonsense question about seawater drifts into incoherent advice; the arithmetic and coding prompts collapse into placeholder tokens or spurious code-like noise rather than a correct numeral or a valid expression. Policy questions produce vague, self-repeating generalities. Together, these traces point to limited grounding and a tendency toward generic fluent filler under small-model fine-tuning, while the sweep above shows how truncation and length caps change whether the failure mode looks like echo, repetition, or abrupt truncation.


## Stage 7. Streamlit Chatbot Interface

The emitted streamlit_chat_app.py binds MODEL_REF to the same BEST_MODEL_DIR archive, unpacks it once, and serves the causal LM on the detected accelerator. Sidebar controls mirror the offline study: tokenizer max_length for the user prefix, max_new_tokens, top_k, top_p, and temperature.

Every user turn is appended as one JSON object per line to chat_log.txt in the application directory (timestamp, decode parameters, user text, assistant text), matching the logging requirement for later qualitative mining alongside generation_interaction_log.txt from Stage 6.


In [34]:
def _lab2_streamlit_app_source() -> str:
    """Emit streamlit_chat_app.py text. __BEST_MODEL_DIR__ is replaced with the training archive path."""
    return """from pathlib import Path
import json
import tarfile
from datetime import datetime, timezone

import streamlit as st
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

APP_DIR = Path(__file__).resolve().parent

st.set_page_config(page_title="Lab 2 Chatbot", page_icon="💬", layout="wide")

MODEL_REF = Path(r"__BEST_MODEL_DIR__")


def resolve_model_dir(p: Path) -> Path:
    if p.is_dir():
        return p
    if p.is_file() and p.name.endswith(".tar.gz"):
        root = p.parent / "_extracted"
        root.mkdir(parents=True, exist_ok=True)
        inner = p.name[: -len(".tar.gz")]
        target = root / inner
        if (target / "config.json").exists():
            return target
        with tarfile.open(p, "r:gz") as tar:
            tar.extractall(path=root)
        return root / inner
    return p


MODEL_DIR = resolve_model_dir(MODEL_REF)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@st.cache_resource
def load_chat_assets():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_DIR)
    device = get_device()
    model.to(device)
    model.eval()
    return tokenizer, model, device


tokenizer, model, device = load_chat_assets()

st.title("Lab 2 LLM Chatbot")
st.caption("Fine-tuned on OpenAssistant OASST1")

with st.sidebar:
    st.subheader("Generation settings")
    encode_max_length = st.slider("tokenizer max_length", 32, 512, 128, 8)
    max_new_tokens = st.slider("max_new_tokens", 16, 256, 80, 8)
    top_k = st.slider("top_k", 10, 100, 50, 5)
    top_p = st.slider("top_p", 0.5, 1.0, 0.92, 0.01)
    temperature = st.slider("temperature", 0.4, 1.4, 0.8, 0.05)

CHAT_LOG = APP_DIR / "chat_log.txt"

if "history" not in st.session_state:
    st.session_state.history = []

for item in st.session_state.history:
    st.chat_message("user").write(item["user"])
    st.chat_message("assistant").write(item["assistant"])

user_input = st.chat_input("Ask something...")
if user_input:
    st.chat_message("user").write(user_input)
    prompt = "User: " + user_input + "\\nAssistant:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=int(encode_max_length),
    ).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            top_k=int(top_k),
            top_p=float(top_p),
            temperature=float(temperature),
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = decoded.split("Assistant:")[-1].strip()
    st.chat_message("assistant").write(answer)

    st.session_state.history.append({"user": user_input, "assistant": answer})

    record = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "encode_max_length": int(encode_max_length),
        "max_new_tokens": int(max_new_tokens),
        "top_k": int(top_k),
        "top_p": float(top_p),
        "temperature": float(temperature),
        "user": user_input,
        "assistant": answer,
    }
    with CHAT_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\\n")
"""

streamlit_app_code = _lab2_streamlit_app_source().replace("__BEST_MODEL_DIR__", str(BEST_MODEL_DIR))
streamlit_path = ROOT / "streamlit_chat_app.py"
streamlit_path.write_text(streamlit_app_code, encoding="utf-8")
streamlit_path


PosixPath('/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-2/streamlit_chat_app.py')

In [35]:
!streamlit run streamlit_chat_app.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://100.82.89.245:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Loading weights: 100%|████████████████████████| 76/76 [00:00<00:00, 5069.05it/s]
Accessing `__path__` from `.models.aria.image_processing_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__path__` from `.models.aria.image_processing_pil_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__path__` from `.models.auto.image_processing_auto`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__path__` from `.models.beit.image_processing_beit`. Returning `__path__` instead. Behavior may be different and this alias will be remo